<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

# Quantize Cosmos3-Nano and Cosmos3-Super to FP8

This notebook converts the Cosmos3-Nano (8B) and Cosmos3-Super (32B) checkpoints to
static-scale **FP8** with NVIDIA TensorRT Model Optimizer. FP8 roughly halves the model's
memory footprint and speeds up inference on NVIDIA GPUs with FP8 tensor cores, while
preserving generation quality.

Nano and Super use the **same recipe on a different checkpoint**. We quantize Nano first,
then Super.

## 1. Prerequisites

Use a Linux machine with an NVIDIA GPU, model access on Hugging Face, and either
`uvx hf@latest auth login` or `HF_TOKEN` set. Quantizing the Super (32B) checkpoints
needs a GPU with enough memory to hold the model in bf16 plus the quantizers; Nano (8B)
is comfortable on a single 48 GB GPU.

You also need free disk for the Hugging Face checkpoint cache and for the FP8 output
(each output is roughly the size of the bf16 source). Point `HF_HOME` at a large volume
in the next step.

> **Headless servers:** if you see `libGL.so.1: cannot open shared object file` when the
> VAE loads, install the system graphics libraries:
>
> ```bash
> apt-get install -y libgl1 libglib2.0-0
> ```

## 2. Configure Paths and Environment

The defaults are relative to this `cosmos` checkout and use the CUDA 13 (or 12.8) Torch
backend depending on your system. Override any of these before running the cell:

```bash
export COSMOS3_QUANTIZE_VENV=/path/to/.venv-cosmos3-quantize
export COSMOS3_TORCH_BACKEND=cu130       # or cu128
export HF_HOME=/path/to/large/huggingface/cache
export OUTPUT_ROOT=/path/to/fp8/outputs
```

In [1]:
from pathlib import Path
import os


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'README.md').exists() and (path / 'cookbooks').exists():
            return path
    return start


def configure_quantize_environment() -> None:
    global COSMOS_ROOT, QUANTIZE_ROOT, COSMOS3_QUANTIZE_VENV
    global COSMOS3_TORCH_BACKEND, OUTPUT_ROOT

    COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
    QUANTIZE_ROOT = COSMOS_ROOT / 'cookbooks' / 'cosmos3' / 'quantization'
    COSMOS3_QUANTIZE_VENV = Path(
        os.environ.get('COSMOS3_QUANTIZE_VENV', COSMOS_ROOT / '.venv-cosmos3-quantize')
    ).resolve()
    COSMOS3_TORCH_BACKEND = os.environ.get('COSMOS3_TORCH_BACKEND', 'cu130')
    OUTPUT_ROOT = Path(os.environ.get('OUTPUT_ROOT', QUANTIZE_ROOT / 'outputs')).resolve()

    os.environ['COSMOS3_QUANTIZE_VENV'] = str(COSMOS3_QUANTIZE_VENV)
    os.environ['COSMOS3_TORCH_BACKEND'] = COSMOS3_TORCH_BACKEND
    os.environ['OUTPUT_ROOT'] = str(OUTPUT_ROOT)
    os.environ.setdefault('UV_LINK_MODE', 'copy')
    os.environ.setdefault('HF_HOME', str(Path.home() / '.cache' / 'huggingface'))
    # Keep the datasets cache writable and separate from the shared hub lock dir.
    os.environ.setdefault('HF_DATASETS_CACHE', str(Path(os.environ['HF_HOME']) / 'datasets'))
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

configure_quantize_environment()
print('COSMOS_ROOT   :', COSMOS_ROOT)
print('QUANTIZE_ROOT :', QUANTIZE_ROOT)
print('OUTPUT_ROOT   :', OUTPUT_ROOT)
print('venv          :', COSMOS3_QUANTIZE_VENV)

COSMOS_ROOT   : <COSMOS>
QUANTIZE_ROOT : <COSMOS>/cookbooks/cosmos3/quantization
OUTPUT_ROOT   : <COSMOS>/cookbooks/cosmos3/quantization/outputs
venv          : <COSMOS>/cookbooks/cosmos3/quantization/.venv-cosmos3-quantize


## 3. Install Dependencies

This creates a dedicated virtual environment with PyTorch, NVIDIA TensorRT Model
Optimizer (ModelOpt — the FP8 quantization engine), Diffusers (VAE + schedulers) and
Transformers (tokenizer), then registers a Jupyter kernel for it. Run this once.

In [ ]:
%%bash
set -euo pipefail

if ! command -v uv >/dev/null 2>&1; then
  echo 'uv is not installed. Install it first: https://docs.astral.sh/uv/getting-started/installation/'
  exit 1
fi

export UV_LINK_MODE="${UV_LINK_MODE:-copy}"
uv venv "$COSMOS3_QUANTIZE_VENV" --python 3.13 --seed --managed-python --allow-existing
source "$COSMOS3_QUANTIZE_VENV/bin/activate"
uv pip install --torch-backend="$COSMOS3_TORCH_BACKEND" \
  "diffusers @ git+https://github.com/huggingface/diffusers.git" \
  "nvidia-modelopt[torch]" \
  accelerate datasets huggingface_hub imageio imageio-ffmpeg ipykernel \
  numpy pillow safetensors torch torchvision transformers

"$COSMOS3_QUANTIZE_VENV/bin/python" -m ipykernel install --user \
  --name cosmos3-quantize \
  --display-name "Cosmos3 Quantize (Python 3.13)"

echo
echo "Installed into: $COSMOS3_QUANTIZE_VENV"
echo "Next: switch this notebook kernel to: Cosmos3 Quantize (Python 3.13)"

## 4. Select the Quantization Kernel

The install cell registers the `Cosmos3 Quantize (Python 3.13)` Jupyter kernel.

**Switch this notebook to that kernel**, then run the restore cell below before
continuing. It can take a moment for a new kernel to appear in the notebook interface.

In [3]:
# Run this cell immediately after switching to the Cosmos3 Quantize kernel.
# It restores the same paths and cache settings as the Configure cell above.
from pathlib import Path
import os


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'README.md').exists() and (path / 'cookbooks').exists():
            return path
    return start


def configure_quantize_environment() -> None:
    global COSMOS_ROOT, QUANTIZE_ROOT, COSMOS3_QUANTIZE_VENV
    global COSMOS3_TORCH_BACKEND, OUTPUT_ROOT

    COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
    QUANTIZE_ROOT = COSMOS_ROOT / 'cookbooks' / 'cosmos3' / 'quantization'
    COSMOS3_QUANTIZE_VENV = Path(
        os.environ.get('COSMOS3_QUANTIZE_VENV', COSMOS_ROOT / '.venv-cosmos3-quantize')
    ).resolve()
    COSMOS3_TORCH_BACKEND = os.environ.get('COSMOS3_TORCH_BACKEND', 'cu130')
    OUTPUT_ROOT = Path(os.environ.get('OUTPUT_ROOT', QUANTIZE_ROOT / 'outputs')).resolve()

    os.environ['COSMOS3_QUANTIZE_VENV'] = str(COSMOS3_QUANTIZE_VENV)
    os.environ['COSMOS3_TORCH_BACKEND'] = COSMOS3_TORCH_BACKEND
    os.environ['OUTPUT_ROOT'] = str(OUTPUT_ROOT)
    os.environ.setdefault('UV_LINK_MODE', 'copy')
    os.environ.setdefault('HF_HOME', str(Path.home() / '.cache' / 'huggingface'))
    # Keep the datasets cache writable and separate from the shared hub lock dir.
    os.environ.setdefault('HF_DATASETS_CACHE', str(Path(os.environ['HF_HOME']) / 'datasets'))
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

configure_quantize_environment()
print('environment restored — OUTPUT_ROOT:', OUTPUT_ROOT)

environment restored — OUTPUT_ROOT: <COSMOS>/cookbooks/cosmos3/quantization/outputs


## 5. Verify GPU and Python Environment

Confirm the kernel sees a GPU and the quantization engine imports cleanly.

In [4]:
import torch, modelopt
print('torch          :', torch.__version__)
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
print('ModelOpt       :', modelopt.__version__)

torch          : 2.13.0+cu130
CUDA available : True
GPU            : NVIDIA B200
ModelOpt       : 0.45.0


## 6. Load the Quantization Toolkit

The cookbook ships the Cosmos3 model and the complete FP8 recipe in `quantize/src/`:
loading, calibration, and export. `quantize_fp8_checkpoint(...)` runs the whole
pipeline for one checkpoint — load the model, **calibrate** it by replaying real
denoising so the quantizer sees representative activations, quantize to FP8, and write a
drop-in checkpoint you can serve.

The two helpers below wrap that call (resolving the Hugging Face checkpoint) and print a
short summary of what was produced.

In [5]:
import sys
sys.path.insert(0, str(QUANTIZE_ROOT))

from huggingface_hub import snapshot_download
from safetensors import safe_open
import json
import src
from src import (quantize_fp8_checkpoint,
                 SHAPE_VIDEO, SHAPE_IMAGE, SHAPE_VIDEO_DEMO, SHAPE_IMAGE_DEMO,
                 SAMPLER_VIDEO_BASE, SAMPLER_IMAGE_BASE, SAMPLER_DISTILLED)

# DEMO=1 (default): one calibration prompt at a small shape, so a run takes minutes.
# Set DEMO=0 for the production shape and 8 calibration prompts (the shipped recipe).
DEMO = os.environ.get('DEMO', '1') == '1'
NUM_SAMPLES = 1 if DEMO else 8
print(f'DEMO={DEMO}  (NUM_SAMPLES={NUM_SAMPLES})')


def resolve_checkpoint(hf_repo, override_env):
    """Local dir from `override_env` if set, else the Hugging Face snapshot."""
    override = os.environ.get(override_env)
    return Path(override) if override else Path(snapshot_download(hf_repo))


def inspect_checkpoint(output_dir):
    """Print a short, human-readable summary of an FP8 checkpoint."""
    output_dir = Path(output_dir)
    tdir = output_dir / 'transformer'
    n_fp8 = n_scale = 0
    example = []
    for shard in sorted(tdir.glob('*.safetensors')):
        with safe_open(str(shard), framework='pt') as h:
            for k in h.keys():
                if k.endswith(('.input_scale', '.weight_scale')):
                    n_scale += 1
                    if k.endswith('.input_scale') and len(example) < 3:
                        example.append((k, float(h.get_tensor(k).reshape(-1)[0])))
                elif k.endswith('.weight') and h.get_slice(k).get_dtype() == 'F8_E4M3':
                    n_fp8 += 1
    qcfg = json.load(open(output_dir / 'hf_quant_config.json'))
    print(f'FP8 checkpoint: {output_dir}')
    print(f"  quant_algo   : {qcfg.get('quant_algo')}  (method={qcfg.get('quant_method')})")
    print(f'  FP8 weights  : {n_fp8}')
    print(f'  scale tensors: {n_scale}')
    for k, v in example:
        print(f'    e.g. {k} = {v:.6g}')

<COSMOS>/cookbooks/cosmos3/quantization/.venv-cosmos3-quantize/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEMO=True  (NUM_SAMPLES=1)


<COSMOS>/cookbooks/cosmos3/quantization/.venv-cosmos3-quantize/lib/python3.13/site-packages/modelopt/torch/__init__.py:51: UserWarning: transformers 5.14.1 is not tested with current version of modelopt and may cause issues. Please install recommended version with `pip install -U nvidia-modelopt[hf]` if working with HF models.
  _warnings.warn(


## Cosmos3-Nano

Nano is the 8B checkpoint — the fastest to quantize and a good first run.

### Quantize

Load Nano, calibrate it, and write the FP8 checkpoint to `OUTPUT_ROOT/nano-fp8`. In DEMO
mode this uses one calibration prompt at a small shape; set `DEMO=0` for the shipped recipe.

In [6]:
input_dir = resolve_checkpoint('nvidia/Cosmos3-Nano', 'C3_NANO_DIR')
output_dir = OUTPUT_ROOT / 'nano-fp8'

quantize_fp8_checkpoint(
    input_dir=input_dir, output_dir=output_dir,
    profile='t2v', sampler=SAMPLER_VIDEO_BASE, shape=SHAPE_VIDEO_DEMO if DEMO else SHAPE_VIDEO,
    num_samples=NUM_SAMPLES,
)

Fetching 68 files:   0%|                                                                             | 0/68 [00:00<?, ?it/s]

Fetching 68 files: 100%|██████████████████████████████████████████████████████████████████| 68/68 [00:00<00:00, 1902.08it/s]

[load] transformer from <HF_HOME>/models--nvidia--Cosmos3-Nano/snapshots/411f42a8fdfb8c5b2583cb8786e0938f49796eaa/transformer (variant=8b)


[load] tokenizer: <HF_HOME>/models--nvidia--Cosmos3-Nano/snapshots/411f42a8fdfb8c5b2583cb8786e0938f49796eaa (local_files_only=True)


[load] scheduler class: UniPCMultistepScheduler
[init] gen_layers=36 profile=t2v sampler='video base (UniPC runtime)'


...

[export] loading bf16 base from <HF_HOME>/models--nvidia--Cosmos3-Nano/snapshots/411f42a8fdfb8c5b2583cb8786e0938f49796eaa/transformer
[export] overlaid 504 fp8 weights + 1008 scales onto bf16 base (dropped 4 vae2llm/llm2vae); 1822 tensors total


[export] wrote 1822 tensors across 4 shard(s)
[export] wrote 1822 tensors (fp8) to <COSMOS>/cookbooks/cosmos3/quantization/outputs/.quantized_transformer_t2v.tmp


[assemble] moving quantized -> <COSMOS>/cookbooks/cosmos3/quantization/outputs/nano-fp8/transformer
[assemble] linking assets -> <HF_HOME>/models--nvidia--Cosmos3-Nano/snapshots/411f42a8fdfb8c5b2583cb8786e0938f49796eaa/assets
[assemble] linking EXPLAINABILITY.md -> <HF_HOME>/models--nvidia--Cosmos3-Nano/snapshots/411f42a8fdfb8c5b2583cb8786e0938f49796eaa/EXPLAINABILITY.md
[assemble] linking SAFETY.md -> <HF_HOME>/models--nvidia--Cosmos3-Nano/snapshots/411f42a8fdfb8c5b2583cb8786e0938f49796eaa/SAFETY.md
[assemble] linking BIAS.md -> <HF_HOME>/models--nvidia--Cosmos3-Nano/snapshots/411f42a8fdfb8c5b2583cb8786e0938f49796eaa/BIAS.md
[assemble] linking .gitattributes -> <HF_HOME>/models--nvidia--Cosmos3-Nano/snapshots/411f42a8fdfb8c5b2583cb8786e0938f49796eaa/.gitattributes
[assemble] linking PRIVACY.md -> <HF_HOME>/models--nvidia--Cosmos3-Nano/snapshots/411f42a8fdfb8c5b2583cb8786e0938f49796eaa/PRIVACY.md
[assemble] linking README.md -> <HF_HOME>/models--nvidia--Cosmos3-Nano/snapshots/411f42a8f

[assemble] wrote model.safetensors.index.json (2173 tensors, 17608584096 bytes)
[assemble] wrote <COSMOS>/cookbooks/cosmos3/quantization/outputs/nano-fp8/hf_quant_config.json for upstream-vLLM discovery
[assemble] drop-in dir ready at <COSMOS>/cookbooks/cosmos3/quantization/outputs/nano-fp8
[done] <COSMOS>/cookbooks/cosmos3/quantization/outputs/nano-fp8


PosixPath('<COSMOS>/cookbooks/cosmos3/quantization/outputs/nano-fp8')

### Inspect the FP8 Checkpoint

A quick look at the drop-in checkpoint just written: the quantization method, how many
weights were converted to FP8, and the per-tensor scales that accompany them.

In [7]:
inspect_checkpoint(OUTPUT_ROOT / 'nano-fp8')

FP8 checkpoint: <COSMOS>/cookbooks/cosmos3/quantization/outputs/nano-fp8
  quant_algo   : FP8  (method=modelopt)
  FP8 weights  : 504
  scale tensors: 1008
    e.g. layers.0.mlp.down_proj.input_scale = 0.015346
    e.g. layers.0.mlp.gate_proj.input_scale = 0.00354004
    e.g. layers.0.mlp.up_proj.input_scale = 0.00354004


## Cosmos3-Super

Super is the 32B checkpoint. The call is identical to Nano — only the checkpoint changes.
This is the heaviest run in the notebook; make sure your GPU has enough memory.

### Quantize

In [8]:
input_dir = resolve_checkpoint('nvidia/Cosmos3-Super', 'C3_SUPER_DIR')
output_dir = OUTPUT_ROOT / 'super-fp8'

quantize_fp8_checkpoint(
    input_dir=input_dir, output_dir=output_dir,
    profile='t2v', sampler=SAMPLER_VIDEO_BASE, shape=SHAPE_VIDEO_DEMO if DEMO else SHAPE_VIDEO,
    num_samples=NUM_SAMPLES,
)

Fetching 88 files:   0%|                                                                             | 0/88 [00:00<?, ?it/s]

Fetching 88 files:   1%|▊                                                                    | 1/88 [00:00<00:30,  2.87it/s]

Fetching 88 files:   9%|██████▎                                                              | 8/88 [00:00<00:06, 12.76it/s]

Fetching 88 files:  11%|███████▋                                                            | 10/88 [00:00<00:05, 13.62it/s]

Fetching 88 files:  14%|█████████▎                                                          | 12/88 [00:00<00:05, 14.75it/s]

...

[export] overlaid 896 fp8 weights + 1792 scales onto bf16 base (dropped 4 vae2llm/llm2vae); 3222 tensors total


[export] wrote 3222 tensors across 14 shard(s)
[export] wrote 3222 tensors (fp8) to <COSMOS>/cookbooks/cosmos3/quantization/outputs/.quantized_transformer_t2v.tmp


[assemble] moving quantized -> <COSMOS>/cookbooks/cosmos3/quantization/outputs/super-fp8/transformer
[assemble] linking assets -> <HF_HOME>/models--nvidia--Cosmos3-Super/snapshots/e0262be9d8f7586bc24c069a2aed2b665bdff266/assets
[assemble] linking .gitattributes -> <HF_HOME>/models--nvidia--Cosmos3-Super/snapshots/e0262be9d8f7586bc24c069a2aed2b665bdff266/.gitattributes
[assemble] linking EXPLAINABILITY.md -> <HF_HOME>/models--nvidia--Cosmos3-Super/snapshots/e0262be9d8f7586bc24c069a2aed2b665bdff266/EXPLAINABILITY.md
[assemble] linking README.md -> <HF_HOME>/models--nvidia--Cosmos3-Super/snapshots/e0262be9d8f7586bc24c069a2aed2b665bdff266/README.md
[assemble] linking PRIVACY.md -> <HF_HOME>/models--nvidia--Cosmos3-Super/snapshots/e0262be9d8f7586bc24c069a2aed2b665bdff266/PRIVACY.md
[assemble] linking BIAS.md -> <HF_HOME>/models--nvidia--Cosmos3-Super/snapshots/e0262be9d8f7586bc24c069a2aed2b665bdff266/BIAS.md
[assemble] linking SAFETY.md -> <HF_HOME>/models--nvidia--Cosmos3-Super/snapshots/e

[assemble] wrote model.safetensors.index.json (3573 tensors, 66818770912 bytes)
[assemble] wrote <COSMOS>/cookbooks/cosmos3/quantization/outputs/super-fp8/hf_quant_config.json for upstream-vLLM discovery
[assemble] drop-in dir ready at <COSMOS>/cookbooks/cosmos3/quantization/outputs/super-fp8
[done] <COSMOS>/cookbooks/cosmos3/quantization/outputs/super-fp8


PosixPath('<COSMOS>/cookbooks/cosmos3/quantization/outputs/super-fp8')

### Inspect the FP8 Checkpoint

A quick look at the drop-in checkpoint just written: the quantization method, how many
weights were converted to FP8, and the per-tensor scales that accompany them.

In [9]:
inspect_checkpoint(OUTPUT_ROOT / 'super-fp8')

FP8 checkpoint: <COSMOS>/cookbooks/cosmos3/quantization/outputs/super-fp8
  quant_algo   : FP8  (method=modelopt)
  FP8 weights  : 896
  scale tensors: 1792
    e.g. layers.0.mlp.down_proj.input_scale = 0.0262277
    e.g. layers.0.mlp.gate_proj.input_scale = 0.00244141
    e.g. layers.0.mlp.up_proj.input_scale = 0.00244141
